In [2]:
#!/usr/bin/env python3
"""
house.lk Full Scraper - ROBUST & CORRECTED VERSION
- Persistent session with retries
- Hardened Price Extraction (Millions, Lakhs, Rupees)
- Breadcrumb & Strict exact-word location matching to prevent district corruption
- Targeted tag parsing to prevent footer/sidebar text contamination
- Safe Fallback parsing engine for newly added or missing suburbs
- Incremental CSV saving & polite delays
"""

import re
import time
import random
import csv
import os
from urllib.parse import urljoin, urlparse
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ----------------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------------
START_PAGE = 1
END_PAGE   = 330
BASE_URL   = "https://house.lk"
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
]
REQUEST_DELAY = (1.5, 3.5)          
OUTPUT_FILE   = "house_lk_corrected.csv"
PARTIAL_FILE  = "house_lk_partial.csv"   
SAVE_EVERY    = 50                                                 

# Expanded and verified Sri Lankan Real Estate Region Map
DISTRICT_MAP = {
    # --- COLOMBO DISTRICT ---
    "colombo": "Colombo", "maharagama": "Colombo", "battaramulla": "Colombo",
    "homagama": "Colombo", "malabe": "Colombo", "mount lavinia": "Colombo",
    "kaduwela": "Colombo", "boralesgamuwa": "Colombo", "dehiwala": "Colombo",
    "nugegoda": "Colombo", "rajagiriya": "Colombo", "kotte": "Colombo",
    "nawala": "Colombo", "kolonnawa": "Colombo", "pelawatta": "Colombo",
    "thalawathugoda": "Colombo", "athurugiriya": "Colombo", "hokandara": "Colombo",
    "pannipitiya": "Colombo", "piliyandala": "Colombo", "moratuwa": "Colombo",
    "ratmalana": "Colombo", "wellawatte": "Colombo", "bambalapitiya": "Colombo",
    "kottawa": "Colombo", "udahamulla": "Colombo", "kahathuduwa": "Colombo",
    "kiriwaththuduwa": "Colombo", "katubedda": "Colombo", "mattegoda": "Colombo",

    # --- GAMPAHA DISTRICT ---
    "gampaha": "Gampaha", "negombo": "Gampaha", "ja ela": "Gampaha", "ja-ela": "Gampaha",
    "kandana": "Gampaha", "wattala": "Gampaha", "kelaniya": "Gampaha", 
    "kiribathgoda": "Gampaha", "kadawatha": "Gampaha", "ragama": "Gampaha",
    "nitambuwa": "Gampaha", "minuwangoda": "Gampaha", "seeduwa": "Gampaha",
    "divulapitiya": "Gampaha", "mirigama": "Gampaha", "veyangoda": "Gampaha",
    "peliyagoda": "Gampaha", "dalugama": "Gampaha",

    # --- KALUTARA DISTRICT ---
    "kalutara": "Kalutara", "panadura": "Kalutara", "horana": "Kalutara",
    "wadduwa": "Kalutara", "beruwala": "Kalutara", "aluthgama": "Kalutara",
    "bandaragama": "Kalutara",

    # --- OTHER MAJOR DISTRICTS ---
    "kandy": "Kandy", "peradeniya": "Kandy", "katugastota": "Kandy", "digana": "Kandy",
    "nawalapitiya": "Kandy",
    "galle": "Galle", "hikkaduwa": "Galle", "unawatuna": "Galle", "koggala": "Galle",
    "ahungalla": "Galle",
    "kurunegala": "Kurunegala", "kuliyapitiya": "Kurunegala",
    "tangalle": "Hambantota", "hambantota": "Hambantota",
    "matara": "Matara", "weligama": "Matara", "mirissa": "Matara",
    "anuradhapura": "Anuradhapura", "polonnaruwa": "Polonnaruwa",
    "ratnapura": "Ratnapura", "badulla": "Badulla", "ella": "Badulla",
    "jaffna": "Jaffna", "matale": "Matale", "dambulla": "Matale",
    "nuwara eliya": "Nuwara Eliya", "trincomalee": "Trincomalee",
    "batticaloa": "Batticaloa", "ampara": "Ampara", "puttalam": "Puttalam",
    "chilaw": "Puttalam", "marawila": "Puttalam", "kegalle": "Kegalle",
    "monaragala": "Monaragala", "mullaitivu": "Mullaitivu", 
    "vavuniya": "Vavuniya", "mannar": "Mannar", "kilinochchi": "Kilinochchi"
}

# Isolated list of native Sri Lankan Districts to handle unrecognized towns dynamically
SRI_LANKA_DISTRICTS = [
    "Colombo", "Gampaha", "Kalutara", "Kandy", "Matale", "Nuwara Eliya", 
    "Galle", "Matara", "Hambantota", "Jaffna", "Kilinochchi", "Mannar", 
    "Vavuniya", "Mullaitivu", "Batticaloa", "Ampara", "Trincomalee", 
    "Kurunegala", "Puttalam", "Anuradhapura", "Polonnaruwa", "Badulla", 
    "Monaragala", "Ratnapura", "Kegalle"
]

# ----------------------------------------------------------------------
# SESSION WITH RETRIES
# ----------------------------------------------------------------------
def get_session_with_retries(retries=3, backoff_factor=2, timeout=30):
    session = requests.Session()
    retry_strategy = Retry(
        total=retries,
        read=retries,
        connect=retries,
        backoff_factor=backoff_factor,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["HEAD", "GET", "OPTIONS"]
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    session.timeout = timeout
    return session

def get_headers():
    return {"User-Agent": random.choice(USER_AGENTS)}

# ----------------------------------------------------------------------
# EXTRACTION HELPERS
# ----------------------------------------------------------------------
def extract_price(text):
    if not text:
        return None
    text = text.replace(",", "")
    
    # 1. Match Millions/M (e.g., "Rs. 45M", "45.5 M", "45 Million")
    m_match = re.search(r"(?:Rs\.?\s*)?([\d\.]+)\s*(?:[Mm]illion|[Mm]\b)", text)
    if m_match:
        return int(float(m_match.group(1)) * 1_000_000)
        
    # 2. Match Lakhs/Laks (e.g., "445 Laks")
    l_match = re.search(r"(?:Rs\.?\s*)?([\d\.]+)\s*(?:[Ll]akhs?|[Ll]aks?)", text, re.I)
    if l_match:
        return int(float(l_match.group(1)) * 100_000)
        
    # 3. Standard Numbers (e.g., "Rs. 45000000")
    r_match = re.search(r"Rs\.?\s*([\d]+)", text, re.I)
    if r_match:
        val = int(r_match.group(1))
        # Safeguard: If the regex caught "Rs. 45" because it was formatted weirdly,
        # it is almost certainly 45 Million. No house in SL costs 45 rupees.
        if val > 0 and val < 1000:
            return val * 1_000_000
        return val
        
    return None

def extract_area(text):
    m = re.search(r"([\d,]+)\s*(?:sq\.?ft|sqft|sq ft)", text, re.I)
    if m:
        return int(m.group(1).replace(",", ""))
    return None

def extract_land_perches(text):
    m = re.search(r"([\d\.]+)\s*perches?", text, re.I)
    if m:
        return float(m.group(1))
    return None

def extract_floor(text):
    m = re.search(r"(\d+)(?:st|nd|rd|th)?\s*floor", text, re.I)
    if m:
        return int(m.group(1))
    if "ground floor" in text.lower():
        return 0
    return None

def whole_word(word, text):
    return re.search(rf'\b{word}\b', text, re.I) is not None

def extract_city_district(soup, url=""):
    # Priority 1: URL Path Parsing (Cleanest, highly reliable)
    if url:
        path = urlparse(url).path.lower().replace("-", " ")
        # Using .items() ensures we can grab both the city key and district value cleanly
        for city, district in sorted(DISTRICT_MAP.items(), key=lambda x: len(x[0]), reverse=True):
            if re.search(rf'\b{re.escape(city)}\b', path):
                return city.title(), district

    # Priority 2: Breadcrumb Parsing (Extremely accurate for classifieds sites)
    breadcrumbs = soup.select(".breadcrumb li, [class*='breadcrumb'] a, .breadcrumbs")
    if breadcrumbs:
        bc_text = " ".join([b.get_text(" ", strip=True) for b in breadcrumbs]).lower()
        for city, district in sorted(DISTRICT_MAP.items(), key=lambda x: len(x[0]), reverse=True):
            if re.search(rf'\b{re.escape(city)}\b', bc_text):
                return city.title(), district

    # Priority 3: Target specific location containers
    loc_elem = soup.select_one(".location, .address, [class*='location'], [class*='address']")
    raw_location_text = loc_elem.get_text(" ", strip=True) if loc_elem else ""

    if raw_location_text:
        loc_lower = raw_location_text.lower()
        
        for city, district in sorted(DISTRICT_MAP.items(), key=lambda x: len(x[0]), reverse=True):
            if re.search(rf'\b{re.escape(city)}\b', loc_lower):
                return city.title(), district
        
        # Priority 4: Safe Fallback Engine
        # Do NOT split by comma. Better to have `None` for city and an accurate District
        for dist in SRI_LANKA_DISTRICTS:
            if re.search(rf'\b{re.escape(dist.lower())}\b', loc_lower):
                return None, dist

    return None, None

# ----------------------------------------------------------------------
# GET LISTING URLS
# ----------------------------------------------------------------------
def get_listing_urls(session, start=1, end=50):
    urls = set()
    for page in range(start, end+1):
        page_url = f"{BASE_URL}/sale/page/{page}/"
        try:
            resp = session.get(page_url, headers=get_headers(), timeout=30)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, "lxml")
            for a in soup.find_all("a", href=True):
                href = a["href"]
                if "/details/" in href:
                    full = urljoin(BASE_URL, href)
                    urls.add(full)
            print(f"Page {page}: collected {len(urls)} unique URLs")
            time.sleep(random.uniform(*REQUEST_DELAY))
        except Exception as e:
            print(f"Error on page {page}: {e}")
    return list(urls)

# ----------------------------------------------------------------------
# SCRAPE A SINGLE PROPERTY
# ----------------------------------------------------------------------
def scrape_property(url, session):
    try:
        resp = session.get(url, headers=get_headers(), timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "lxml")
        text = soup.get_text(" ", strip=True)
        text_lower = text.lower()
        title_tag = soup.find("h1")
        title = title_tag.get_text(strip=True) if title_tag else ""

        data = {"url": url, "title": title}

        # Price Extraction
        price_elem = soup.select_one(".price, .property-price, .amount, [class*='price']")
        price_text = price_elem.get_text(strip=True) if price_elem else ""
        data["price"] = extract_price(price_text) or extract_price(text)

        # Basic Stats Extraction
        m = re.search(r"(\d+)\s*Bedrooms?", text, re.I)
        data["bedrooms"] = int(m.group(1)) if m else None

        m = re.search(r"(\d+)\s*Bathrooms?", text, re.I)
        data["bathrooms"] = int(m.group(1)) if m else None

        data["area_sqft"] = extract_area(text)
        data["land_perches"] = extract_land_perches(text)

        m = re.search(r"(\d+)\s*Car\s*parking", text, re.I)
        data["parking"] = int(m.group(1)) if m else None

        m = re.search(r"Age of Building\s*(\d+)", text, re.I)
        data["building_age"] = int(m.group(1)) if m else None

        m = re.search(r"Year\s*[Bb]uilt\s*(\d{4})", text)
        data["year_built"] = int(m.group(1)) if m else None

        m = re.search(r"(\d+)\s*[Ss]torey", text)
        data["storeys"] = int(m.group(1)) if m else None

        data["floor_level"] = extract_floor(text_lower)

        # Property Type Detection
        if whole_word("house", title) or whole_word("bungalow", title):
            property_type = "House"
        elif whole_word("apartment", title) or whole_word("flat", title) or whole_word("condo", title):
            property_type = "Apartment"
        elif whole_word("villa", title):
            property_type = "Villa"
        elif whole_word("land", title):
            property_type = "Land"
        elif whole_word("commercial", title):
            property_type = "Commercial Property"
        else:
            house_count = len(re.findall(r'\bhouse\b', text_lower))
            apt_count = len(re.findall(r'\bapartment\b', text_lower)) + len(re.findall(r'\bflat\b', text_lower))
            if house_count > apt_count and house_count > 0:
                property_type = "House"
            elif apt_count > 0:
                property_type = "Apartment"
            elif whole_word("villa", text_lower):
                property_type = "Villa"
            elif whole_word("land", text_lower):
                property_type = "Land"
            elif whole_word("commercial", text_lower):
                property_type = "Commercial Property"
            else:
                property_type = None
        data["property_type"] = property_type

        # Furnished Status
        if "unfurnished" in text_lower:
            data["furnished"] = "Unfurnished"
        elif "furnished" in text_lower:
            data["furnished"] = "Furnished"
        else:
            data["furnished"] = None

        # Amenities (Booleans)
        data["has_pool"] = "pool" in text_lower or "swimming" in text_lower
        data["has_ac"] = "air condition" in text_lower or "a/c" in text_lower or "ac" in text_lower
        data["has_garden"] = "garden" in text_lower or "landscaped" in text_lower
        data["has_cctv"] = "cctv" in text_lower
        data["has_hot_water"] = "hot water" in text_lower
        data["has_servant_room"] = "servant" in text_lower or "maid" in text_lower
        data["has_balcony"] = "balcony" in text_lower
        data["has_garage"] = "garage" in text_lower

        # Location Parsing
        loc_elem = soup.select_one(".location, .address, [class*='location'], [class*='address']")
        data["location_raw"] = loc_elem.get_text(strip=True) if loc_elem else None
        
        # Scrape precise City and District values safely
        city, district = extract_city_district(soup, url)
        data["city"] = city
        data["district"] = district

        # Description Extraction
        desc_elem = soup.select_one(".description, .details, [class*='desc']")
        data["description"] = desc_elem.get_text(strip=True) if desc_elem else None

        return data

    except Exception as e:
        print(f"Failed on {url}: {e}")
        return None

# ----------------------------------------------------------------------
# SAVE PARTIAL RESULTS
# ----------------------------------------------------------------------
def save_partial(data_list, filename):
    if not data_list:
        return
    df = pd.DataFrame(data_list)
    df.to_csv(filename, index=False)

# ----------------------------------------------------------------------
# MAIN RUNNER
# ----------------------------------------------------------------------
def main():
    session = get_session_with_retries(retries=3, backoff_factor=2, timeout=30)
    
    print("Collecting property URLs...")
    urls = get_listing_urls(session, START_PAGE, END_PAGE)
    print(f"Found {len(urls)} properties. Starting extraction...")
    
    all_data = []
    for idx, url in enumerate(tqdm(urls), 1):
        prop = scrape_property(url, session)
        if prop:
            all_data.append(prop)
        
        if len(all_data) % SAVE_EVERY == 0 and len(all_data) > 0:
            save_partial(all_data, PARTIAL_FILE)
            print(f"\n[Partial Save] Logged {len(all_data)} properties to {PARTIAL_FILE}")
        
        time.sleep(random.uniform(*REQUEST_DELAY))
    
    if not all_data:
        print("No data collected. Exiting.")
        return
    
    df = pd.DataFrame(all_data)
    
    # Pricing Category Logic
    def price_cat(price):
        if price is None:
            return None
        if price < 15_000_000:
            return "Budget"
        elif price < 40_000_000:
            return "Mid_Range"
        elif price < 100_000_000:
            return "Premium"
        else:
            return "Luxury"
    df["price_category"] = df["price"].apply(price_cat)
    
    df = df.drop_duplicates(subset=["url"])
    df = df[df["price"].notna()]
    
    # Structure Clean Columns
    cols = [
        "url", "title", "price", "price_category",
        "city", "district", "location_raw",
        "bedrooms", "bathrooms", "area_sqft", "land_perches",
        "property_type", "furnished", "parking", "storeys", "floor_level",
        "building_age", "year_built",
        "has_pool", "has_ac", "has_garden", "has_cctv", "has_hot_water",
        "has_servant_room", "has_balcony", "has_garage",
        "description"
    ]
    df = df[[c for c in cols if c in df.columns]]
    
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"\n✅ Completed! Saved {len(df)} properties to {OUTPUT_FILE}")
    
    if os.path.exists(PARTIAL_FILE):
        os.remove(PARTIAL_FILE)

if __name__ == "__main__":
    main()

Page 1: collected 30 unique URLs
Page 2: collected 59 unique URLs
Page 3: collected 89 unique URLs
Page 4: collected 119 unique URLs
Page 5: collected 149 unique URLs
Page 6: collected 179 unique URLs
Page 7: collected 209 unique URLs
Page 8: collected 239 unique URLs
Page 9: collected 269 unique URLs
Page 10: collected 299 unique URLs
Page 11: collected 329 unique URLs
Page 12: collected 359 unique URLs
Page 13: collected 389 unique URLs
Page 14: collected 419 unique URLs
Page 15: collected 449 unique URLs
Page 16: collected 479 unique URLs
Page 17: collected 509 unique URLs
Page 18: collected 539 unique URLs
Page 19: collected 569 unique URLs
Page 20: collected 599 unique URLs
Page 21: collected 629 unique URLs
Page 22: collected 659 unique URLs
Page 23: collected 689 unique URLs
Page 24: collected 719 unique URLs
Page 25: collected 749 unique URLs
Page 26: collected 779 unique URLs
Page 27: collected 809 unique URLs
Page 28: collected 839 unique URLs
Page 29: collected 869 unique UR

  0%|          | 49/9809 [03:59<10:14:16,  3.78s/it]


[Partial Save] Logged 50 properties to house_lk_partial.csv


  1%|          | 99/9809 [08:09<13:55:46,  5.16s/it]


[Partial Save] Logged 100 properties to house_lk_partial.csv


...


[Partial Save] Logged 9750 properties to house_lk_partial.csv


100%|█████████▉| 9800/9809 [12:30:14<00:59,  6.62s/it]


[Partial Save] Logged 9800 properties to house_lk_partial.csv


100%|██████████| 9809/9809 [12:30:51<00:00,  4.59s/it]



✅ Completed! Saved 9808 properties to house_lk_corrected.csv
